# Chord + refine geodesic gallery

Renders the chord + refinement gallery and the geodesic-length and
waypoint-distance table for the toy-manifold geodesics runs.

Runs are read from
`results/toy_manifolds_test/geodesics/{manifold}/seed_{seed}/` (set
`RESULTS_ROOT_NAME` below; switch it to `toy_manifolds` for the paper runs).
Figure helpers live in `notebooks/helpers/`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from IPython.display import Image, Markdown, display


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / "experiments" / "toy_manifolds").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root (expected experiments/toy_manifolds/)."
    )


REPO_ROOT = _find_repo_root()

for _p in (REPO_ROOT, REPO_ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from notebooks.helpers import geodesics as geo
from notebooks.helpers.style import apply_paper_style

apply_paper_style()

RESULTS_ROOT_NAME = "toy_manifolds"  # "toy_manifolds" for the paper runs
RESULTS_ROOT = REPO_ROOT / "results" / RESULTS_ROOT_NAME
OUT_DIR = REPO_ROOT / "notebooks" / "figures" / "toy_manifolds"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFOLD_NAMES = ("hemisphere", "saddle", "paraboloid")
SEED = 42
RUN_ID = f"seed_{SEED}"


def available_seeds(name: str) -> list[int]:
    root = RESULTS_ROOT / "geodesics" / name
    return sorted(
        int(path.name.removeprefix("seed_"))
        for path in root.glob("seed_*")
        if path.is_dir()
    )


def show_fig(path: Path) -> None:
    png = path.with_suffix(".png")
    display(Image(filename=str(png if png.is_file() else path)))


print("REPO_ROOT =", REPO_ROOT)
print("OUT_DIR   =", OUT_DIR)
print("RESULTS   =", RESULTS_ROOT)
print("RUN_ID    =", RUN_ID)
for _name in MANIFOLD_NAMES:
    print(f"{_name:<12} {geo.geodesics_dir(RESULTS_ROOT, _name, RUN_ID)}")


In [ ]:
path, rows = geo.fig_chord_refine_geodesics_gallery(
    OUT_DIR,
    RESULTS_ROOT,
    RUN_ID,
    n_show=5,
    text_scale=1.5,
    manifolds=list(MANIFOLD_NAMES),
)
print("wrote", path)
for r in rows:
    print(r["manifold"])
show_fig(path)


In [ ]:
run_ids = [
    f"seed_{seed}"
    for seed in sorted(
        {seed for name in MANIFOLD_NAMES for seed in available_seeds(name)}
    )
]
table_rows = geo.collect_chord_refine_table(
    RESULTS_ROOT, list(MANIFOLD_NAMES), run_ids
)
display(Markdown(geo.format_chord_refine_table_markdown(table_rows)))
print(geo.format_chord_refine_table_latex(table_rows))
